# 3.2 线性注意力与亚二次复杂度架构

## 什么是线性注意力？

标准注意力的$O(n^2)$复杂度限制了长序列推理。线性注意力通过kernel化或分解将复杂度降至$O(n)$，状态空间模型（SSM）通过隐状态递推实现$O(n)$推理。

## 为什么需要线性注意力？

1. **长序列瓶颈**：标准注意力的$O(n^2)$复杂度使得序列长度超过一定阈值后，计算量和内存急剧增长。
2. **端侧内存限制**：端侧设备无法存储$n \times n$的注意力矩阵，线性复杂度是必需的。
3. **实时性要求**：流式处理（如语音识别）需要恒定时间的每步推理。

## 线性注意力的核心思想

标准注意力：$\text{Attn}(Q,K,V) = \text{softmax}(QK^T/\sqrt{d})V$，先算$QK^T$（$O(n^2)$）

线性注意力：$\text{LinAttn}(Q,K,V) = \frac{\phi(Q)(\phi(K)^T V)}{\phi(Q)(\phi(K)^T \mathbf{1})}$，先算$\phi(K)^T V$（$O(nd^2)$）

其中$\phi$是特征映射函数，将softmax的指数函数替换为可分解的核函数。当$d \ll n$时，复杂度从$O(n^2)$降至$O(nd^2)$。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")

---
### 线性注意力（Linear Attention）

#### 什么是线性注意力？

将softmax注意力中的指数核替换为可分解的特征映射$\phi$，改变计算顺序使复杂度从$O(n^2)$降至$O(nd^2)$。

#### 为什么线性注意力有效？

1. **计算顺序变换**：标准注意力先算$QK^T$（$O(n^2)$），线性注意力先算$\phi(K)^T V$（$O(nd^2)$）
2. **递推计算**：$\phi(K)^T V$可以用递推更新，每步$O(d^2)$
3. **长序列友好**：当$n \gg d$时，$O(nd^2) \ll O(n^2)$

#### 线性注意力的数学原理

标准注意力：$\text{Attn}(Q,K,V) = \text{softmax}(QK^T/\sqrt{d})V$

线性注意力：
$$\text{LinAttn}(Q,K,V) = \frac{\phi(Q)(\phi(K)^T V)}{\phi(Q)(\phi(K)^T \mathbf{1})}$$

其中$\phi$为特征映射函数，常见选择：
- **ELU+1**：$\phi(x) = \text{ELU}(x) + 1$
- **ReLU**：$\phi(x) = \text{ReLU}(x)$
- **Performer**：$\phi(x) = \frac{1}{\sqrt{m}}e^{iWx}$（随机正交投影）

关键：$\phi(K)^T V \in \mathbb{R}^{d \times d}$与序列长度$n$无关，可递推更新

In [ ]:
class LinearAttention(nn.Module):
    """线性注意力：使用ELU+1作为特征映射"""
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.eps = 1e-6

    def feature_map(self, x):
        """特征映射 φ(x) = elu(x) + 1"""
        return F.elu(x) + 1

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

        q_feat = self.feature_map(q)
        k_feat = self.feature_map(k)

        kv = torch.einsum('bhnd,bhne->bhde', k_feat, v)
        qkv = torch.einsum('bhnd,bhde->bhne', q_feat, kv)
        k_sum = k_feat.sum(dim=2, keepdim=True)
        normalizer = torch.einsum('bhnd,bhkd->bhn', q_feat, k_sum.expand(-1, -1, N, -1))
        normalizer = normalizer.unsqueeze(-1).clamp(min=self.eps)

        out = qkv / normalizer
        out = out.transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

class StandardAttention(nn.Module):
    """标准Softmax注意力"""
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

dim, n_heads = 128, 4
linear_attn = LinearAttention(dim, n_heads)
standard_attn = StandardAttention(dim, n_heads)

print(f"=== 线性注意力 vs 标准注意力 ===")
for seq_len in [64, 128, 256, 512, 1024]:
    x = torch.randn(1, seq_len, dim)
    with torch.no_grad():
        start = time.time()
        for _ in range(20):
            _ = standard_attn(x)
        std_time = time.time() - start

        start = time.time()
        for _ in range(20):
            _ = linear_attn(x)
        lin_time = time.time() - start

    print(f"seq={seq_len:<6} 标准={std_time:.4f}s 线性={lin_time:.4f}s 比值={std_time/lin_time:.2f}x")

print(f"\n线性注意力优势: 复杂度O(Nd²) vs 标准O(N²d)")
print(f"当N >> d时，线性注意力显著更快")

---
### 状态空间模型（State Space Models, SSM）

#### 什么是SSM？

通过隐状态的线性递推建模序列依赖，推理时每步仅需$O(d)$计算和$O(d)$内存，不受序列长度影响。Mamba引入选择性机制，使SSM具备输入依赖的建模能力。

#### 为什么SSM有效？

1. **线性递推**：每步仅需$O(d)$计算，远低于注意力的$O(nd)$
2. **并行训练**：虽然推理是递推的，但训练时可通过卷积模式并行
3. **长程记忆**：通过选择机制（Mamba）可以高效捕获长程依赖

#### SSM的数学原理

连续时间SSM：
$$\dot{h}(t) = Ah(t) + Bx(t), \quad y(t) = Ch(t) + Dx(t)$$

离散化（零阶保持）：
$$h_t = \bar{A}h_{t-1} + \bar{B}x_t, \quad y_t = Ch_t + Dx_t$$

其中：
- $h_t \in \mathbb{R}^{d}$：隐状态
- $\bar{A} = e^{\Delta A}$：离散化状态转移矩阵
- $\bar{B} = (\Delta A)^{-1}(e^{\Delta A} - I) \cdot \Delta B$：离散化输入矩阵
- $\Delta$：步长（可学习）

Mamba的选择机制：$B, C, \Delta$依赖于输入$x_t$，使模型能选择性地传播或遗忘信息

In [ ]:
class SimpleSSM(nn.Module):
    """简化SSM层：线性递推 + 输出投影"""
    def __init__(self, dim, state_dim=16):
        super().__init__()
        self.dim = dim
        self.state_dim = state_dim
        self.A = nn.Parameter(torch.randn(state_dim, state_dim) * 0.01)
        self.B = nn.Parameter(torch.randn(state_dim, dim) * 0.01)
        self.C = nn.Parameter(torch.randn(dim, state_dim) * 0.01)
        self.D = nn.Parameter(torch.ones(dim))
        self.dt_proj = nn.Linear(dim, state_dim)

    def forward(self, x):
        """SSM前向：逐步递推"""
        B_batch, N, D = x.shape
        h = torch.zeros(B_batch, self.state_dim, device=x.device)
        outputs = []

        for t in range(N):
            x_t = x[:, t, :]
            dt = F.softplus(self.dt_proj(x_t)).unsqueeze(-1)
            dA = torch.exp(self.A.unsqueeze(0) * dt)
            dB = self.B.unsqueeze(0) * dt
            h = h * dA.squeeze(-1) + (x_t.unsqueeze(-1) * dB).squeeze(-1).transpose(-1, -2).sum(dim=-1, keepdim=True).squeeze(-1)
            y_t = h @ self.C.T + x_t * self.D
            outputs.append(y_t)

        return torch.stack(outputs, dim=1)

class SelectiveSSMBlock(nn.Module):
    """选择性SSM块 (Mamba风格)"""
    def __init__(self, dim, state_dim=16, expand_factor=2):
        super().__init__()
        self.dim = dim
        self.inner_dim = dim * expand_factor
        self.in_proj = nn.Linear(dim, self.inner_dim * 2, bias=False)
        self.conv1d = nn.Conv1d(self.inner_dim, self.inner_dim, kernel_size=3, padding=1, groups=self.inner_dim)
        self.ssm = SimpleSSM(self.inner_dim, state_dim)
        self.out_proj = nn.Linear(self.inner_dim, dim, bias=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        projected = self.in_proj(x)
        x1, x2 = projected.chunk(2, dim=-1)
        x1 = self.conv1d(x1.transpose(1, 2)).transpose(1, 2)
        x1 = F.silu(x1)
        x1 = self.ssm(x1)
        out = x1 * F.silu(x2)
        return self.out_proj(out) + residual

dim = 64
ssm_block = SelectiveSSMBlock(dim, state_dim=16, expand_factor=2)
x = torch.randn(2, 32, dim)
out = ssm_block(x)

print(f"=== SSM (Mamba风格) ===")
print(f"输入: {x.shape}")
print(f"输出: {out.shape}")
print(f"参数量: {sum(p.numel() for p in ssm_block.parameters()):,}")

print(f"\n--- SSM vs Attention 推理复杂度对比 ---")
print(f"{'序列长度':<12} {'Attention O(N²)':<20} {'SSM O(N)':<20}")
print("-" * 52)
for N in [128, 256, 512, 1024, 2048, 4096, 8192]:
    attn_ops = N * N
    ssm_ops = N * 16 * 64
    print(f"{N:<12} {attn_ops:<20,} {ssm_ops:<20,}")

print(f"\nSSM核心优势:")
print(f"1. 推理时O(1)每步（固定状态大小），不受序列长度影响")
print(f"2. 无需KV Cache，内存占用恒定")
print(f"3. 选择性机制使SSM具备输入依赖的建模能力")

### 混合架构（Hybrid Architecture）

#### 什么是混合架构？

将注意力层和SSM/线性注意力层交替堆叠，在关键位置保留精确注意力，其余位置使用高效线性层。如Jamba、Zamba等模型。

#### 为什么混合架构有效？

1. **精确+高效**：注意力层提供精确的长程依赖建模，SSM层提供高效的序列处理
2. **精度接近纯注意力**：少量注意力层（如每4层1个）即可保持接近纯注意力模型的精度
3. **推理效率高**：大部分层是SSM，推理速度和内存占用接近纯SSM模型

#### 混合架构的数学原理

混合层的计算：
$$y = \begin{cases} \text{SSM}(x) & \text{if layer } l \mod s \neq 0 \\ \text{Attention}(x) + \text{SSM}(x) & \text{if layer } l \mod s = 0 \end{cases}$$

其中$s$为注意力间隔（如$s=4$表示每4层一个注意力层）。

计算复杂度：$O(\frac{L}{s} n^2 d + L \cdot n d^2)$，当$s$较大时接近$O(Lnd^2)$

In [ ]:
class HybridBlock(nn.Module):
    """混合块：可选择Attention或SSM"""
    def __init__(self, dim, block_type='ssm', n_heads=4, state_dim=16):
        super().__init__()
        self.block_type = block_type
        if block_type == 'attention':
            self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
            self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.SiLU(), nn.Linear(dim*4, dim))
            self.ln1 = nn.LayerNorm(dim)
            self.ln2 = nn.LayerNorm(dim)
        else:
            self.ssm = SelectiveSSMBlock(dim, state_dim=state_dim)

    def forward(self, x):
        if self.block_type == 'attention':
            h = self.ln1(x)
            h, _ = self.attn(h, h, h)
            x = x + h
            x = x + self.ffn(self.ln2(x))
            return x
        else:
            return self.ssm(x)

class HybridModel(nn.Module):
    """混合架构：SSM层为主，每隔几层插入Attention层"""
    def __init__(self, dim=64, n_layers=8, attn_interval=4, n_heads=4, state_dim=16):
        super().__init__()
        self.blocks = nn.ModuleList()
        for i in range(n_layers):
            if (i + 1) % attn_interval == 0:
                self.blocks.append(HybridBlock(dim, 'attention', n_heads, state_dim))
            else:
                self.blocks.append(HybridBlock(dim, 'ssm', n_heads, state_dim))

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

dim = 64
hybrid = HybridModel(dim=dim, n_layers=8, attn_interval=4, n_heads=4, state_dim=16)

n_attn = sum(1 for b in hybrid.blocks if b.block_type == 'attention')
n_ssm = sum(1 for b in hybrid.blocks if b.block_type == 'ssm')

print(f"=== 混合架构 (Jamba风格) ===")
print(f"总层数: 8")
print(f"Attention层: {n_attn} (第4,8层)")
print(f"SSM层: {n_ssm} (其余层)")
print(f"\n架构可视化:")
for i, block in enumerate(hybrid.blocks):
    tag = '🔵 Attention' if block.block_type == 'attention' else '🟢 SSM'
    print(f"  Layer {i+1}: {tag}")

x = torch.randn(2, 32, dim)
out = hybrid(x)
print(f"\n输入: {x.shape}, 输出: {out.shape}")

print(f"\n--- 不同序列长度的推理时间对比 ---")
attn_only = nn.ModuleList([HybridBlock(dim, 'attention', 4) for _ in range(8)])
ssm_only = nn.ModuleList([HybridBlock(dim, 'ssm', 4, 16) for _ in range(8)])

for seq_len in [64, 128, 256, 512]:
    x = torch.randn(1, seq_len, dim)
    with torch.no_grad():
        start = time.time()
        for block in attn_only:
            x_a = block(x)
        attn_t = time.time() - start

        start = time.time()
        for block in ssm_only:
            x_s = block(x)
        ssm_t = time.time() - start

        start = time.time()
        for block in hybrid.blocks:
            x_h = block(x)
        hybrid_t = time.time() - start

    print(f"  N={seq_len:<5} Attention={attn_t:.4f}s SSM={ssm_t:.4f}s Hybrid={hybrid_t:.4f}s")

print(f"\n混合架构优势: 在关键位置保留精确注意力，其余位置用SSM降低复杂度")

### 线性RNN架构：RWKV风格

#### 什么是RWKV？

RWKV使用线性递推替代传统RNN的非线性激活，结合时间混合（Time Mixing）和通道混合（Channel Mixing）交替，实现$O(1)$推理复杂度和可并行训练。

#### 为什么RWKV有效？

1. **O(1)推理**：每步仅需维护固定大小的隐状态，不随序列增长
2. **可并行训练**：训练时将递推展开为并行扫描（parallel scan）
3. **注意力兼容**：时间混合模块使用类似注意力的加权求和，但权重是递推计算的

#### RWKV的数学原理

**时间混合（WKV注意力）**：
$$wkv_t = \frac{\sum_{i=1}^{t-1} e^{-(t-1-i)w + u_i} \cdot v_i}{\sum_{i=1}^{t-1} e^{-(t-1-i)w + u_i}}$$

递推形式：
$$wkv_t = \frac{e^{u_{t-1}} \cdot v_{t-1} + e^{-w} \cdot a_{t-1}}{e^{u_{t-1}} + e^{-w} \cdot b_{t-1}}$$

其中：
- $w$：时间衰减因子（可学习），控制历史信息的重要性
- $u_i$：位置奖励（可学习），给当前位置额外权重
- $a_t, b_t$：递推累积量，$a_t = e^{u_{t-1}} v_{t-1} + e^{-w} a_{t-1}$
- 输出：$o_t = \sigma(r_t) \cdot (wkv_t \cdot W_o + x_t \cdot W_{o,r})$

In [ ]:
class RWKVTimeMixing(nn.Module):
    """RWKV时间混合：线性递推的注意力替代
    核心思想：使用可学习的衰减因子w，实现类似注意力的信息聚合，
    但递推复杂度为O(1)而非O(n²)。"""
    def __init__(self, dim, n_heads=4):
        super().__init__()
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.time_decay = nn.Parameter(torch.randn(n_heads, self.head_dim) * 0.1)
        self.time_first = nn.Parameter(torch.randn(n_heads, self.head_dim) * 0.1)
        self.key = nn.Linear(dim, dim, bias=False)
        self.value = nn.Linear(dim, dim, bias=False)
        self.receptance = nn.Linear(dim, dim, bias=False)
        self.output = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        r = torch.sigmoid(self.receptance(x)).view(B, N, self.n_heads, self.head_dim)
        k = self.key(x).view(B, N, self.n_heads, self.head_dim)
        v = self.value(x).view(B, N, self.n_heads, self.head_dim)

        w = torch.exp(-torch.abs(self.time_decay))
        u = self.time_first

        outputs = []
        state = torch.zeros(B, self.n_heads, self.head_dim, self.head_dim, device=x.device)
        for t in range(N):
            kt = k[:, t, :, :]
            vt = v[:, t, :, :]
            rt = r[:, t, :, :]
            wkv = (state @ kt.unsqueeze(-1)).squeeze(-1) * u + (state.sum(dim=-1) * vt)
            state = state * w.unsqueeze(0).unsqueeze(-1) + kt.unsqueeze(-1) * vt.unsqueeze(-2)
            out_t = rt * wkv
            outputs.append(out_t)

        out = torch.stack(outputs, dim=1).reshape(B, N, C)
        return self.output(out)

class RWKVChannelMixing(nn.Module):
    """RWKV通道混合：FFN的线性递推版本"""
    def __init__(self, dim, hidden_factor=4):
        super().__init__()
        hidden = dim * hidden_factor
        self.key = nn.Linear(dim, hidden, bias=False)
        self.value = nn.Linear(hidden, dim, bias=False)
        self.receptance = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        r = torch.sigmoid(self.receptance(x))
        k = torch.relu(self.key(x)) ** 2
        return r * self.value(k)

dim = 128
rwkv_tm = RWKVTimeMixing(dim, n_heads=4)
rwkv_cm = RWKVChannelMixing(dim)

x = torch.randn(2, 64, dim)
with torch.no_grad():
    tm_out = rwkv_tm(x)
    cm_out = rwkv_cm(x)

print(f"=== RWKV线性RNN架构 ===")
print(f"输入: {x.shape}")
print(f"时间混合输出: {tm_out.shape}")
print(f"通道混合输出: {cm_out.shape}")

tm_params = sum(p.numel() for p in rwkv_tm.parameters())
cm_params = sum(p.numel() for p in rwkv_cm.parameters())
print(f"\n参数量:")
print(f"  时间混合: {tm_params:,}")
print(f"  通道混合: {cm_params:,}")

print(f"\nRWKV vs Transformer vs SSM:")
comparisons = [
    ('Transformer', 'O(n²)', 'O(n)', '并行训练, 精确注意力'),
    ('SSM (Mamba)', 'O(n)', 'O(1)', '线性训练, O(1)推理'),
    ('RWKV', 'O(n)', 'O(1)', '线性训练, O(1)推理, 类注意力'),
]
print(f"  {'架构':<15} {'训练复杂度':<12} {'推理复杂度':<12} {'特点':<25}")
print("  " + "-" * 64)
for name, train, infer, feat in comparisons:
    print(f"  {name:<15} {train:<12} {infer:<12} {feat:<25}")